In [ ]:
import pandas as pd
import numpy as np

In [ ]:
BASE = 'PLACE THE DATASET FOLDER PATH HERE'


# Loading Data

In [ ]:
demo = pd.read_csv(BASE + 'DEMO25Q4.txt', delimiter='$', low_memory=False)
drug = pd.read_csv(BASE + 'DRUG25Q4.txt', delimiter='$', low_memory=False)
reac = pd.read_csv(BASE + 'REAC25Q4.txt', delimiter='$', low_memory=False)
outc = pd.read_csv(BASE + 'OUTC25Q4.txt', delimiter='$', low_memory=False)
indi = pd.read_csv(BASE + 'INDI25Q4.txt', delimiter='$', low_memory=False)
ther = pd.read_csv(BASE + 'THER25Q4.txt', delimiter='$', low_memory=False)

# Age Cleanup

In [ ]:
def normalize_age(row):
    if pd.isna(row['age']):
        return None
    cod = str(row['age_cod']).upper()
    if cod == 'YR':  return row['age']
    if cod == 'MON': return row['age'] / 12
    if cod == 'WK':  return row['age'] / 52
    if cod == 'DY':  return row['age'] / 365
    if cod == 'DEC': return row['age'] * 10
    return None

demo['age_years']   = demo.apply(normalize_age, axis=1)
demo['missing_age'] = demo['age_years'].isna().astype(int)
demo['age_years']   = demo['age_years'].fillna(demo['age_years'].median())
demo = demo[demo['age_years'] > 0]
demo['age_years']   = demo['age_years'].astype(int)

# Weight Cleanup

In [ ]:
demo['wt_kg'] = demo['wt']

lbs_mask  = (demo['wt_cod'] == 'LBS') | (
    (demo['reporter_country'] == 'US') & (demo['wt'] <= 550)
)
gram_mask = demo['wt'] >= 1000
kg_mask   = (demo['wt_cod'] == 'KG') & (demo['wt'] <= 250)

demo.loc[lbs_mask,  'wt_kg'] = demo.loc[lbs_mask,  'wt'] * 0.453592
demo.loc[gram_mask, 'wt_kg'] = demo.loc[gram_mask, 'wt'] / 1000
demo.loc[kg_mask,   'wt_kg'] = demo.loc[kg_mask,   'wt']
demo.loc[(demo['wt_kg'] > 250) | (demo['wt_kg'] < 2), 'wt_kg'] = np.nan

demo['missing_wt'] = demo['wt_kg'].isna().astype(int)

Keeping latest case version only

In [ ]:
demographic = demo[[
    'primaryid', 'caseid', 'caseversion',
    'sex', 'age_years', 'missing_age',
    'wt_kg', 'missing_wt',
    'reporter_country', 'occr_country',
    'rept_cod', 'occp_cod',
    'event_dt', 'rept_dt'
]].copy()
demographic = (
    demographic
    .sort_values('caseversion')
    .drop_duplicates('caseid', keep='last')
)

In [ ]:
print(f"  demographic rows after dedup: {len(demographic)}")

  demographic rows after dedup: 385133


# Drug Cleanup

In [ ]:
drug = drug[drug['role_cod'].isin(['PS', 'SS'])].copy()
drug['drugname'] = drug['drugname'].str.upper().str.strip()
drug['prod_ai']  = drug['prod_ai'].str.upper().str.strip()
drug['route']    = drug['route'].str.upper().str.strip()
drug['dose_form']= drug['dose_form'].str.upper().str.strip()
drug['dose_freq']= drug['dose_freq'].str.upper().str.strip()

print(f"  drug rows after PS/SS filter: {len(drug)}")

  drug rows after PS/SS filter: 1124526


# Reaction Cleanup

In [ ]:
reac = reac[['primaryid', 'pt']].copy()
reac['pt'] = reac['pt'].str.upper().str.strip()

# OUTC Cleanup

In [ ]:
SERIOUS_CODES = {'DE', 'LT', 'HO', 'DS', 'CA', 'RI'}
outc['serious'] = outc['outc_cod'].isin(SERIOUS_CODES).astype(int)

SEVERITY_RANK = {'DE': 5, 'LT': 4, 'HO': 3, 'DS': 3, 'CA': 3, 'RI': 3, 'OT': 1}
outc['severity'] = outc['outc_cod'].map(SEVERITY_RANK).fillna(0).astype(int)

# INDI CLEANING  (comorbidities / indications)

In [ ]:
indi = indi[['primaryid', 'indi_pt']].copy()
indi['indi_pt'] = indi['indi_pt'].str.upper().str.strip()
# Drop generic uninformative entries
indi = indi[~indi['indi_pt'].isin(['PRODUCT USED FOR UNKNOWN INDICATION',
                                   'UNKNOWN'])]

# THER CLEANING  (time on drug before AE)

In [ ]:
def parse_faers_date(series):
    """FAERS dates are floats like 20240101.0 → datetime"""
    return pd.to_datetime(
        series.dropna().astype(int).astype(str),
        format='%Y%m%d',
        errors='coerce'
    )

In [ ]:
ther['start_dt_parsed'] = parse_faers_date(ther['start_dt'])

In [ ]:
ther = ther.merge(
    demographic[['primaryid', 'event_dt']],
    on='primaryid', how='left'
)
ther['event_dt_parsed'] = parse_faers_date(ther['event_dt'])

In [ ]:
dur = ther['dur'].values
cod = ther['dur_cod'].str.upper().fillna('').values

dur_days = np.where(cod == 'YR',  dur * 365,
           np.where(cod == 'MON', dur * 30,
           np.where(cod == 'WK',  dur * 7,
           np.where(cod == 'DY',  dur,
           np.where(cod == 'HR',  dur / 24,
           np.nan)))))

delta_days = (ther['event_dt_parsed'] - ther['start_dt_parsed']).dt.days.values
delta_days = np.where(delta_days >= 0, delta_days, np.nan)

**Priority: reported dur → computed delta**

In [ ]:
ther['days_on_drug'] = np.where(~np.isnan(dur_days), dur_days, delta_days)

# Aggregating

In [ ]:
def pipe_unique(series):
    return '|'.join(series.dropna().astype(str).unique())

def pipe_all(series):
    return '|'.join(series.dropna().astype(str))

Outcome

In [ ]:
outc_agg = (
    outc.groupby('primaryid', sort=False)
    .agg(
        outc_cod = ('outc_cod', pipe_unique),
        serious  = ('serious',  'max'),     # 1 if ANY serious outcome
        severity = ('severity', 'max'),     # highest severity score
    )
    .reset_index()
)

REAC

In [ ]:
reac_agg = (
    reac.groupby('primaryid', sort=False)['pt']
    .agg(pipe_unique)
    .reset_index()
    .rename(columns={'pt': 'reactions'})
)

DRUG

In [ ]:
drug_agg = (
    drug.groupby('primaryid', sort=False)
    .agg(
        drugname   = ('drugname',  pipe_unique),
        role_cod   = ('role_cod',  pipe_unique),
        route      = ('route',     pipe_unique),
        dose_amt   = ('dose_amt',  pipe_all),
        dose_unit  = ('dose_unit', pipe_unique),
        dose_form  = ('dose_form', pipe_unique),
        dose_freq  = ('dose_freq', pipe_unique),
        dechal     = ('dechal',    pipe_unique),
        rechal     = ('rechal',    pipe_unique),
        drug_count = ('drugname',  'count'),    # polypharmacy count
    )
    .reset_index()
)

INDI (comorbidities)

In [ ]:
indi_agg = (
    indi.groupby('primaryid', sort=False)['indi_pt']
    .agg(pipe_unique)
    .reset_index()
    .rename(columns={'indi_pt': 'indications'})
)

THER (time on drug before AE)

In [ ]:
ther_agg = (
    ther.groupby('primaryid', sort=False)['days_on_drug']
    .agg(
        days_on_drug_min  = 'min',
        days_on_drug_max  = 'max',
        days_on_drug_mean = 'mean',
    )
    .reset_index()
)

# MASTER DATASET

In [ ]:
master = demographic.copy()
master = master.merge(outc_agg,  on='primaryid', how='left')
master = master.merge(reac_agg,  on='primaryid', how='left')
master = master.merge(drug_agg,  on='primaryid', how='left')
master = master.merge(indi_agg,  on='primaryid', how='left')
master = master.merge(ther_agg,  on='primaryid', how='left')

# Fill missing target
master['serious']  = master['serious'].fillna(0).astype(int)
master['severity'] = master['severity'].fillna(0).astype(int)

# Derived feature: report lag (days between AE and report)
master['event_dt_parsed'] = parse_faers_date(master['event_dt'])
master['rept_dt_parsed']  = parse_faers_date(master['rept_dt'])
master['report_lag_days'] = (
    master['rept_dt_parsed'] - master['event_dt_parsed']
).dt.days.clip(lower=0)

Drop raw date columns (parsed versions used for features)

In [ ]:
master.drop(columns=['event_dt_parsed', 'rept_dt_parsed'], inplace=True)

In [ ]:
print(f"\n Master shape: {master.shape}")
print(f"   Columns: {master.columns.tolist()}")
print(f"\n   Serious (1): {master['serious'].sum():,}")
print(f"   Non-serious (0): {(master['serious']==0).sum():,}")
print(f"   Class balance: {master['serious'].mean():.1%} serious")


 Master shape: (385133, 33)
   Columns: ['primaryid', 'caseid', 'caseversion', 'sex', 'age_years', 'missing_age', 'wt_kg', 'missing_wt', 'reporter_country', 'occr_country', 'rept_cod', 'occp_cod', 'event_dt', 'rept_dt', 'outc_cod', 'serious', 'severity', 'reactions', 'drugname', 'role_cod', 'route', 'dose_amt', 'dose_unit', 'dose_form', 'dose_freq', 'dechal', 'rechal', 'drug_count', 'indications', 'days_on_drug_min', 'days_on_drug_max', 'days_on_drug_mean', 'report_lag_days']

   Serious (1): 108,782
   Non-serious (0): 276,351
   Class balance: 28.2% serious


# Saving

In [ ]:
out_path = BASE + 'FAERS_master_set.csv'
master.to_csv(out_path, index=False)
print(f"\n Saved → {out_path}")


💾 Saved → /content/drive/MyDrive/Dataset/FAERS_master_set.csv
